In [31]:
import pandas as pd

In [32]:
# Load dataset
df = pd.read_csv("../preparation/shiftvalues.csv")
print(df.shape)
print(df.columns)

(16578, 39)
Index(['GAME_DATE', 'GAME_ID', 'SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION',
       'TEAM_NAME', 'MATCHUP', 'WL', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M',
       'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST',
       'STL', 'BLK', 'TOV', 'PF', 'PTS', 'PLUS_MINUS', 'VIDEO_AVAILABLE',
       'OPP', 'WIN', 'HOME', 'PREV_WIN', 'PREV_PTS', 'PREV_PLUSMINUS',
       'WIN_STREAK', 'LOSE_STREAK', 'DAYS_REST', 'IS_BACK_TO_BACK'],
      dtype='str')


In [33]:
# Make sure data is sorted chronologically within each team/season
df = df.sort_values(["TEAM_ID", "SEASON_ID", "GAME_DATE"]).reset_index(drop=True)

# Shift WIN by 1 so today's game is never included in today's win%
df["WIN_SHIFTED"] = df.groupby(["TEAM_ID", "SEASON_ID"])["WIN"].shift(1)

# Expanding mean of the shifted column = win% using only prior games this season
df["SEASON_WIN_PCT"] = (
    df.groupby(["TEAM_ID", "SEASON_ID"])["WIN_SHIFTED"]
    .expanding()
    .mean()
    .reset_index(level=[0, 1], drop=True)
)

# Drop the helper column
df = df.drop(columns=["WIN_SHIFTED"])

In [34]:
# opp_winpct_df = df[["GAME_ID", "TEAM_ABBREVIATION", "SEASON_WIN_PCT"]].copy()
# opp_winpct_df = opp_winpct_df.rename(
#     columns={"TEAM_ABBREVIATION": "OPP", "SEASON_WIN_PCT": "OPP_SEASON_WIN_PCT"}
# )
# df = pd.merge(df, opp_winpct_df, on=["GAME_ID", "OPP"], how="left")

In [35]:
print(df.columns)

Index(['GAME_DATE', 'GAME_ID', 'SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION',
       'TEAM_NAME', 'MATCHUP', 'WL', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M',
       'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST',
       'STL', 'BLK', 'TOV', 'PF', 'PTS', 'PLUS_MINUS', 'VIDEO_AVAILABLE',
       'OPP', 'WIN', 'HOME', 'PREV_WIN', 'PREV_PTS', 'PREV_PLUSMINUS',
       'WIN_STREAK', 'LOSE_STREAK', 'DAYS_REST', 'IS_BACK_TO_BACK',
       'SEASON_WIN_PCT'],
      dtype='str')


In [36]:
# Sort date
df = df.sort_values(by='GAME_DATE', ascending=True)

In [37]:
def calculate_elo(df, k=20, initial_rating=1500):
    df = df.sort_values(["GAME_DATE", "GAME_ID"]).reset_index(drop=True)
    
    ratings = {}  # TEAM_ID -> current rating
    pre_game_elo = []  # this team's rating BEFORE this game
    
    for _, row in df.iterrows():
        team = row["TEAM_ID"]
        current_rating = ratings.get(team, initial_rating)
        pre_game_elo.append(current_rating)
    
    df["ELO"] = pre_game_elo
    
    # Now actually update ratings game-by-game using both teams' results
    ratings = {}
    elo_dict = {}
    for game_id, group in df.groupby("GAME_ID"):
        if len(group) != 2:
            continue
        t1, t2 = group.iloc[0], group.iloc[1]
        r1 = ratings.get(t1["TEAM_ID"], initial_rating)
        r2 = ratings.get(t2["TEAM_ID"], initial_rating)
        
        elo_dict[(game_id, t1["TEAM_ID"])] = r1
        elo_dict[(game_id, t2["TEAM_ID"])] = r2
        
        expected1 = 1 / (1 + 10 ** ((r2 - r1) / 400))
        actual1 = t1["WIN"]
        
        ratings[t1["TEAM_ID"]] = r1 + k * (actual1 - expected1)
        ratings[t2["TEAM_ID"]] = r2 + k * ((1 - actual1) - (1 - expected1))
    
    df["ELO"] = df.apply(lambda r: elo_dict.get((r["GAME_ID"], r["TEAM_ID"])), axis=1)
    return df

df = calculate_elo(df)

In [38]:
import numpy as np

def calculate_elo(df, k=20, initial_rating=1500):
    df = df.sort_values(["GAME_DATE", "GAME_ID"]).reset_index(drop=True)

    ratings = {}
    elo_dict = {}

    for game_id, group in df.groupby("GAME_ID"):
        if len(group) != 2:
            continue

        t1, t2 = group.iloc[0], group.iloc[1]
        r1 = ratings.get(t1["TEAM_ID"], initial_rating)
        r2 = ratings.get(t2["TEAM_ID"], initial_rating)

        elo_dict[(game_id, t1["TEAM_ID"])] = r1
        elo_dict[(game_id, t2["TEAM_ID"])] = r2

        expected1 = 1 / (1 + 10 ** ((r2 - r1) / 400))
        actual1 = t1["WIN"]

        # Margin of victory, from t1's perspective (negative if t1 lost)
        point_diff = t1["PTS"] - t2["PTS"]
        winner_elo_diff = (r1 - r2) if point_diff > 0 else (r2 - r1)

        # 538-style MOV multiplier:
        # - grows with margin (log-scaled, so a 30-pt win isn't "3x" a 10-pt win)
        # - shrinks when the favorite wins as expected (autocorrelation dampener)
        mov_multiplier = np.log(abs(point_diff) + 1) * (2.2 / (winner_elo_diff * 0.001 + 2.2))

        k_adjusted = k * mov_multiplier

        ratings[t1["TEAM_ID"]] = r1 + k_adjusted * (actual1 - expected1)
        ratings[t2["TEAM_ID"]] = r2 + k_adjusted * ((1 - actual1) - (1 - expected1))

    df["ELO"] = df.apply(lambda r: elo_dict.get((r["GAME_ID"], r["TEAM_ID"])), axis=1)
    return df

df = calculate_elo(df)

In [39]:
def add_league_position(df):
    df = df.sort_values(["SEASON_ID", "GAME_DATE", "GAME_ID"]).reset_index(drop=True)
    
    positions = []
    
    # Process one season at a time
    for season_id, season_df in df.groupby("SEASON_ID"):
        season_df = season_df.sort_values("GAME_DATE")
        
        # Track cumulative wins/losses per team as we move through the season
        wins = {}
        losses = {}
        
        for game_id, game_group in season_df.groupby("GAME_ID", sort=False):
            # Rank BEFORE updating with today's result (pre-game standing)
            standings = pd.DataFrame({
                "TEAM_ID": list(wins.keys()),
                "W": [wins[t] for t in wins],
                "L": [losses[t] for t in wins],
            })
            
            if len(standings) > 0:
                standings["WIN_PCT"] = standings["W"] / (standings["W"] + standings["L"])
                standings["RANK"] = standings["WIN_PCT"].rank(ascending=False, method="min")
                rank_lookup = dict(zip(standings["TEAM_ID"], standings["RANK"]))
            else:
                rank_lookup = {}
            
            for _, row in game_group.iterrows():
                team = row["TEAM_ID"]
                positions.append({
                    "GAME_ID": game_id,
                    "TEAM_ID": team,
                    "LEAGUE_POSITION": rank_lookup.get(team, np.nan)  # NaN if team's first game
                })
            
            # Now update wins/losses with today's results, for future games
            for _, row in game_group.iterrows():
                team = row["TEAM_ID"]
                wins[team] = wins.get(team, 0) + (1 if row["WIN"] == 1 else 0)
                losses[team] = losses.get(team, 0) + (1 if row["WIN"] == 0 else 0)
    
    position_df = pd.DataFrame(positions)
    df = df.merge(position_df, on=["GAME_ID", "TEAM_ID"], how="left")
    return df

df = add_league_position(df)

In [40]:
# opp_elo_df = df[["GAME_ID", "TEAM_ABBREVIATION", "ELO"]].copy()
# opp_elo_df = opp_elo_df.rename(
#     columns={"TEAM_ABBREVIATION": "OPP", "ELO": "OPP_ELO"}
# )
# df = pd.merge(df, opp_elo_df, on=["GAME_ID", "OPP"], how="left")

In [41]:
df = df.sort_values(["TEAM_ID", "GAME_DATE"]).reset_index(drop=True)

for col in ["PTS", "PLUS_MINUS", "FG_PCT", "REB", "AST", "TOV"]:
    df[f"ROLL3_{col}"] = (
        df.groupby("TEAM_ID")[col]
        .shift(1)                      # exclude current game — same anti-leakage rule
        .rolling(window=3, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )

In [42]:
# Sort date
df = df.sort_values(by='GAME_DATE', ascending=True)

In [43]:
# Remove first game of every team
# df = df.groupby(['TEAM_ID', 'SEASON_ID']).apply(lambda x: x.iloc[1:]).reset_index(drop=True)

In [44]:
print(df.columns)

Index(['GAME_DATE', 'GAME_ID', 'SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION',
       'TEAM_NAME', 'MATCHUP', 'WL', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M',
       'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST',
       'STL', 'BLK', 'TOV', 'PF', 'PTS', 'PLUS_MINUS', 'VIDEO_AVAILABLE',
       'OPP', 'WIN', 'HOME', 'PREV_WIN', 'PREV_PTS', 'PREV_PLUSMINUS',
       'WIN_STREAK', 'LOSE_STREAK', 'DAYS_REST', 'IS_BACK_TO_BACK',
       'SEASON_WIN_PCT', 'ELO', 'LEAGUE_POSITION', 'ROLL3_PTS',
       'ROLL3_PLUS_MINUS', 'ROLL3_FG_PCT', 'ROLL3_REB', 'ROLL3_AST',
       'ROLL3_TOV'],
      dtype='str')


In [46]:
# Save to csv
df.to_csv("../preparation/teammetrics.csv", index=False)
print(df.shape)

(16578, 48)
